In [49]:
import torch
import torch.nn as nn
from torch_geometric.nn import GATv2Conv


class GraphEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, edge_dim):
        super(GraphEncoder, self).__init__()
        self.out_channels = out_channels
        self.conv1 = GATv2Conv(
            in_channels, hidden_channels, edge_dim=edge_dim, heads=4, concat=True
        )
        # Second GATv2Conv layer
        self.conv2 = GATv2Conv(
            hidden_channels * 4, out_channels, edge_dim=edge_dim, heads=1, concat=False
        )

    def forward(self, x, edge_index, edge_attr):
        # x: Node features
        # edge_index: Edge connections
        # edge_attr: Edge features
        x = self.conv1(x, edge_index, edge_attr)
        x = torch.relu(x)
        x = self.conv2(x, edge_index, edge_attr)
        return x


class TemporalModel(nn.Module):
    def __init__(self, graph_encoder, hidden_dim, num_layers, out_channel):
        super(TemporalModel, self).__init__()
        self.graph_encoder = graph_encoder
        self.rnn = nn.LSTM(
            input_size=graph_encoder.out_channels,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_dim, out_channel)  # Predict for the ego vehicle

    def forward(self, graph_sequence):
        # graph_sequence: List of graphs over time
        graph_embeddings = []
        for graph in graph_sequence:
            x, edge_index, edge_attr = graph.x, graph.edge_index, graph.edge_attr
            graph_embedding = self.graph_encoder(x, edge_index, edge_attr)
            graph_embeddings.append(graph_embedding)

        # Stack embeddings over time
        graph_embeddings = torch.stack(
            graph_embeddings, dim=1
        )  # Shape: [batch_size, seq_len, embedding_dim]

        # Pass through RNN
        rnn_out, _ = self.rnn(graph_embeddings)

        # Predict for the ego vehicle (assume it's the first node in the graph)
        ego_vehicle_embedding = rnn_out[:, -1, :]  # Last timestamp
        output = self.fc(ego_vehicle_embedding)
        return output

In [45]:
import torch
from torch_geometric.data import Data

# Define node features (4 nodes, 2 features each)
x = torch.tensor([
    [1.0, 2.0],  # Node 0 features
    [3.0, 4.0],  # Node 1 features
    [5.0, 6.0],  # Node 2 features
    [7.0, 8.0]   # Node 3 features
], dtype=torch.float)

# Define edge connections (source, target)
edge_index = torch.tensor([
    [0, 1, 2, 3, 1],  # Source nodes
    [1, 2, 3, 0, 3]   # Target nodes
], dtype=torch.long)

# Define edge features (5 edges, 1 feature each)
edge_attr = torch.tensor([
    [0.1],  # Edge 0->1
    [0.2],  # Edge 1->2
    [0.3],  # Edge 2->3
    [0.4],  # Edge 3->0
    [0.5]   # Edge 1->3
], dtype=torch.float)

# Create the graph object
graph = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

graph_sequence=[]
graph_sequence.append(graph)
graph_sequence.append(graph)

encoder=GraphEncoder(2,128,16,1)
model=TemporalModel(encoder,16,1,1)
model.forward(graph_sequence)

tensor([[-0.1262],
        [-0.1282],
        [-0.1278],
        [-0.1271]], grad_fn=<AddmmBackward0>)

In [54]:
import torch
from torch_geometric.data import Data

def create_graph_array(train_d: dict) -> list:
    graph_list = []
    for k, v in train_d.items():
        nodes = torch.tensor([], dtype=torch.float)
        edge_indexes = torch.tensor([[], []], dtype=torch.long)
        edge_features = torch.tensor([], dtype=torch.float)
        k = str(k)
        v = dict(v)
        ego = v.pop("ego_vehicle")
        ego_pos = ego["position"]
        ego_wayp = ego["waypoint_location"]
        ego_node = torch.tensor(
            [
                [
                    ego_pos["x"],
                    ego_pos["y"],
                    ego_pos["z"],
                    ego_wayp["x"],
                    ego_wayp["y"],
                    ego_wayp["z"],
                    ego["speed"],
                    0,
                    0,
                    0,
                ]
            ],
            dtype=torch.float,
        )
        nodes = torch.cat((nodes, ego_node), dim=0)
        index = 1
        for key, value in v.items():
            exo_pos = value["Position"]
            exo_rotation = value["Rotation"]
            exo_velocity = value["Velocity"]
            exo_rel_pos = value["relative_position"][0]
            exo_rel_dir = value["relative_direction"]
            exo_node = torch.tensor(
                [
                    [
                        exo_pos["x"],
                        exo_pos["y"],
                        exo_pos["z"],
                        exo_rotation["pitch"],
                        exo_rotation["yaw"],
                        exo_rotation["roll"],
                        value["Speed"],
                        exo_velocity["x"],
                        exo_velocity["y"],
                        exo_velocity["z"],
                    ]
                ],
                dtype=torch.float,
            )
            nodes = torch.cat((nodes, exo_node), dim=0)
            e_index = torch.tensor([[0, index], [index, 0]], dtype=torch.long)
            edge_indexes = torch.cat((edge_indexes, e_index), dim=1)
            e_attribute = torch.tensor(
                [
                    [
                        exo_rel_pos["x"],
                        exo_rel_pos["y"],
                        exo_rel_pos["z"],
                        exo_rel_dir["x"],
                        exo_rel_dir["y"],
                        exo_rel_dir["z"],
                    ]
                ],
                dtype=torch.float,
            )
            edge_features = torch.cat((edge_features, e_attribute), dim=0)
            edge_features = torch.cat((edge_features, e_attribute), dim=0)

            index += 1
            graph = Data(x=nodes, edge_index=edge_indexes, edge_attr=edge_features)
            graph_list.append(graph)

    return graph_list

In [62]:
import torch
from torch_geometric.nn import GATv2Conv, global_mean_pool

class GATv2SequenceModel(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, edge_dim, out_channels, heads=1):
        super(GATv2SequenceModel, self).__init__()
        
        # GATv2 Layers
        self.gat1 = GATv2Conv(in_channels, hidden_channels, heads=heads, edge_dim=edge_dim)
        self.gat2 = GATv2Conv(hidden_channels * heads, hidden_channels, heads=heads, edge_dim=edge_dim)
        
        # GRU for Sequence Processing
        self.gru = torch.nn.GRU(hidden_channels * heads, hidden_channels, batch_first=True,bidirectional=True)
        
        # Fully Connected Layer for Predictions
        self.fc = torch.nn.Linear(hidden_channels, out_channels)

    def forward(self, data_list):
        graph_embeddings = []
        
        for data in data_list:
            x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr
            
            # GATv2 Layers
            x = self.gat1(x, edge_index, edge_attr).relu()
            x = self.gat2(x, edge_index, edge_attr).relu()
            
            # Aggregate node features into graph embeddings
            graph_embedding = global_mean_pool(x, data.batch)  # [batch_size, hidden_channels]
            graph_embeddings.append(graph_embedding)
        
        # Stack graph embeddings into a sequence tensor
        graph_sequence = torch.stack(graph_embeddings, dim=1)  # [batch_size, seq_len, hidden_channels]
        
        # Pass through GRU
        _, h_n = self.gru(graph_sequence)
        
        # Final output layer
        out = self.fc(h_n[-1])  # Use the final GRU state
        return out

In [92]:
from torch.utils.data import Dataset, DataLoader

class GraphSequenceDataset(Dataset):
    def __init__(self, graph_sequences, labels):
        """
        Initializes the dataset for graph sequences.
        Args:
            graph_sequences (list): A list of graph sequences (each sequence is a list of `Data` objects).
            labels (list): A list of integer labels corresponding to each sequence.
        """
        self.graph_sequences = graph_sequences  # List of sequences (each sequence is a list of graphs)
        self.labels = labels  # List of labels for each sequence

    def __len__(self):
        """
        Returns the number of sequences in the dataset.
        """
        return len(self.graph_sequences)

    def __getitem__(self, idx):
        """
        Returns a single graph sequence and its label.
        Args:
            idx (int): Index of the sequence to fetch.
        Returns:
            (list, int): A tuple containing the graph sequence and its label.
        """
        graph_sequence = self.graph_sequences[idx]
        label = self.labels[idx]
        return graph_sequence, label



In [93]:
import json
import networkx as nx
from torch_geometric.utils import to_networkx
with open("formatted.json","r") as f:
    test_set=json.load(f)
a=create_graph_array(test_set)
# model=GATv2SequenceModel(10,16,6,5,1)
# model.forward(a[0:20])
# Create the dataset
graph_sequences=[a[i:i+20] for i in range(0, len(a), 20)]
sequence_labels=[1,1,1,1,2,3,1,2,1,1,3,2,1,1,1,1,1]

dataset = GraphSequenceDataset(graph_sequences, sequence_labels)

# Create a DataLoader
batch_size = 1  # Example batch size (you can adjust this)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [97]:
import torch
import torch.nn as nn
import torch.optim as optimizer
from torch.utils.data import Dataset, DataLoader

# PyTorch Geometric imports
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GATv2Conv, global_mean_pool

# For debugging or general utilities (optional)
import numpy as np  # If you need to manipulate arrays
import random  # For reproducibility or shuffling sequences

In [98]:
for epoch in range(10):  # Number of epochs
    model.train()
    total_loss = 0

    for batch in dataloader:
        graph_sequences, labels = batch

        # Move data to GPU if available
        if torch.cuda.is_available():
            model = model.cuda()
            graph_sequences = [[graph.cuda() for graph in seq] for seq in graph_sequences]
            labels = labels.cuda()

        # Forward pass
        optimizer.zero_grad()
        outputs = model(graph_sequences)  # Pass the batch of sequences through the model

        # Compute loss
        loss = criterion(outputs, labels)
        total_loss += loss.item()

        # Backward pass and optimization step
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {total_loss / len(dataloader)}")

TypeError: default_collate: batch must contain tensors, numpy arrays, numbers, dicts or lists; found <class 'torch_geometric.data.data.Data'>

In [ ]:
from torch_geometric.data import Data

# Example graph data

seq_len = 20
batch_size = 4

# Create dataset and dataloader
dataset = GraphSequenceDataset(a, seq_len)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
# Define model
model = GATv2SequenceModel(in_channels=10, hidden_channels=16, edge_dim=8, out_channels=5)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = torch.nn.CrossEntropyLoss()

# Training loop
for epoch in range(10):
    model.train()
    total_loss = 0

    for batch in loader:
        optimizer.zero_grad()

        # Batch is a list of sequences; each sequence is a list of graphs
        output = model(batch)

        labels = torch.randint(0, 5, (batch_size,))  # Example dummy labels
        loss = criterion(output, labels)

        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()

    print(f"Epoch {epoch}, Loss: {total_loss}")

RuntimeError: mat1 and mat2 shapes cannot be multiplied (34x6 and 8x32)